# MLOps Training 2026/2027 — Task 2
## Notebook 5 — Feature Engineering & Leakage-Proof Preprocessing Pipeline

### Objective
In this notebook, we construct an end-to-end feature preprocessing pipeline using `scikit-learn`:
1. **Prevent Data Leakage**: Explicitly drop post-purchase features (e.g., actual delivery date, review scores/comments, order status), driven by the prediction-time feature contract produced in Notebook 4.
2. **Engineer Legitimate Checkout Features**: 
   - `estimated_delivery_days`: Estimated duration between purchase and estimated delivery.
   - Temporal calendar features: Year, month, day, weekday, hour, is_weekend.
3. **Assemble ColumnTransformer**:
   - Numeric Pipeline: `SimpleImputer(strategy='median', add_indicator=True)` + `StandardScaler()`
   - Categorical Pipeline: `SimpleImputer(strategy='constant')` + `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=..., max_categories=...)` — rare categories are grouped instead of the whole column being dropped.
4. **Strict Zero-Leakage Fitting**:
   - Fit pipeline **ONLY on `X_train`**.
   - Transform `X_validation` and `X_test` without refitting.
5. **Persist Artifacts**: Save the fitted preprocessor (`joblib`), each of its fitted sub-objects individually, transformed feature matrices (`.npz`), and target arrays (`.npy`).

### Input Artifacts
- `artifacts/notebook_03/train.parquet`
- `artifacts/notebook_03/validation.parquet`
- `artifacts/notebook_03/test.parquet`
- `artifacts/notebook_04/prediction_time_feature_table.csv` (optional — the EDA leakage/review decision; falls back to a hard-coded mirror of it if not present)

### Output Artifacts
- `artifacts/notebook_05/preprocessor.joblib` (combined transformer — load this in production)
- `artifacts/notebook_05/numeric_imputer.joblib`, `numeric_scaler.joblib`, `categorical_imputer.joblib`, `categorical_encoder.joblib` (each fitted object saved individually)
- `artifacts/notebook_05/X_train.npz`, `X_validation.npz`, `X_test.npz`
- `artifacts/notebook_05/y_train.npy`, `y_validation.npy`, `y_test.npy`
- `artifacts/notebook_05/feature_config.json`, `feature_names.json`, `feature_list.csv`


## 1. Import Libraries & Configure Paths

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import save_npz

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Dynamic project paths
artifact_dir = Path.cwd() / "artifacts" / "notebook_05"
artifact_dir.mkdir(parents=True, exist_ok=True)

input_dir = Path.cwd() / "artifacts" / "notebook_03"
print("Input artifact dir:", input_dir.resolve())
print("Output artifact dir:", artifact_dir.resolve())

Input artifact dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_03
Output artifact dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_05


## 2. Load Split Artifacts

In [2]:
train_df = pd.read_parquet(input_dir / "train.parquet")
validation_df = pd.read_parquet(input_dir / "validation.parquet")
test_df = pd.read_parquet(input_dir / "test.parquet")

TARGET = "is_late"
y_train = train_df[TARGET].copy()
y_validation = validation_df[TARGET].copy()
y_test = test_df[TARGET].copy()

X_train = train_df.drop(columns=[TARGET]).copy()
X_validation = validation_df.drop(columns=[TARGET]).copy()
X_test = test_df.drop(columns=[TARGET]).copy()

print("✓ Data loaded:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_validation.shape}, y_val:   {y_validation.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")

✓ Data loaded:
  X_train: (67533, 27), y_train: (67533,)
  X_val:   (14471, 27), y_val:   (14471,)
  X_test:  (14472, 27), y_test:  (14472,)


## 3. Engineer Legitimate Checkout-Time Features

Before dropping raw timestamp columns, we compute:
1. **`estimated_delivery_days`**: Difference between `order_estimated_delivery_date` and `order_purchase_timestamp` (known at checkout).
2. **Purchase Calendar Features**: `purchase_year`, `purchase_month`, `purchase_day`, `purchase_weekday`, `purchase_hour`, `is_weekend`.

In [3]:
def engineer_checkout_features(df):
    df = df.copy()
    
    # Ensure timestamps
    for col in ["order_purchase_timestamp", "order_estimated_delivery_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            
    # Legitimate estimated delivery window feature
    if "order_estimated_delivery_date" in df.columns and "order_purchase_timestamp" in df.columns:
        df["estimated_delivery_days"] = (
            df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
        ).dt.total_seconds() / (24 * 3600)
        
    # Purchase calendar features
    if "order_purchase_timestamp" in df.columns:
        ts = df["order_purchase_timestamp"]
        df["purchase_year"] = ts.dt.year
        df["purchase_month"] = ts.dt.month
        df["purchase_day"] = ts.dt.day
        df["purchase_weekday"] = ts.dt.weekday
        df["purchase_hour"] = ts.dt.hour
        df["is_weekend"] = ts.dt.weekday.isin([5, 6]).astype(int)
        
    return df

X_train = engineer_checkout_features(X_train)
X_validation = engineer_checkout_features(X_validation)
X_test = engineer_checkout_features(X_test)

print("✓ Engineered checkout features:")
print("  estimated_delivery_days, purchase_year, purchase_month, purchase_day, purchase_weekday, purchase_hour, is_weekend")

✓ Engineered checkout features:
  estimated_delivery_days, purchase_year, purchase_month, purchase_day, purchase_weekday, purchase_hour, is_weekend


## 4. Prevent Data Leakage & Drop Uninformative Identifiers

Rather than duplicating Notebook 4's leakage judgement calls by hand, we load
its `prediction_time_feature_table.csv` (the explicit prediction-time audit)
and drop every column marked:
- **`POTENTIAL_LEAKAGE`** — post-outcome variables (e.g. `order_delivered_customer_date`, `delivery_delay_days`, `average_review_score`).
- **`REVIEW`** — columns whose prediction-time availability isn't guaranteed. This includes `order_status` (can change after placement) and `order_approved_at` (only safe if the prediction timestamp is guaranteed to be after approval); we conservatively exclude both. It also flags the raw ID columns.

If the Notebook 4 artifact isn't available in this environment, we fall back
to a hard-coded mirror of that same audit so this notebook can still run
standalone. Raw timestamp columns are dropped last, now that the checkout
features have already been extracted from them.


In [4]:
# Load the prediction-time feature contract produced by Notebook 4 (EDA) so
# leakage/review decisions live in ONE place instead of being duplicated (and
# potentially drifting) across notebooks.
eda_artifact_dir = Path.cwd() / "artifacts" / "notebook_04"
feature_contract_path = eda_artifact_dir / "prediction_time_feature_table.csv"
if not feature_contract_path.exists():
    feature_contract_path = eda_artifact_dir / "prediction_time_audit.csv"

id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
]

# Raw timestamps: dropped now that checkout-time features have been extracted
# from them above; they are prediction-time-safe (per Notebook 4) but are not
# themselves useful model inputs once decomposed.
raw_timestamp_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

# Fallback mirror of Notebook 4's audit, used only if that artifact isn't
# present in this environment.
FALLBACK_LEAKAGE_COLUMNS = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days",
    "review_score",
    "review_count",
    "average_review_score",
    "review_comment_title",
    "review_comment_message",
]
# order_status can change after order placement, and order_approved_at is
# only safe if the prediction timestamp is guaranteed to fall after
# approval — Notebook 4 marks both "REVIEW" and we conservatively exclude
# them rather than assume either is available at prediction time.
FALLBACK_REVIEW_COLUMNS = [
    "order_status",
    "order_approved_at",
]

if feature_contract_path.exists():
    contract = pd.read_csv(feature_contract_path)
    leakage_columns = contract.loc[
        contract["prediction_time_status"] == "POTENTIAL_LEAKAGE", "column"
    ].tolist()
    review_columns = [
        c for c in contract.loc[
            contract["prediction_time_status"] == "REVIEW", "column"
        ].tolist()
        if c not in id_columns
    ]
    print(f"✓ Loaded prediction-time feature contract from: {feature_contract_path}")
else:
    leakage_columns = FALLBACK_LEAKAGE_COLUMNS
    review_columns = FALLBACK_REVIEW_COLUMNS
    print(
        "⚠ EDA feature contract not found at "
        f"{feature_contract_path} — falling back to the hard-coded "
        "leakage/review lists mirrored from Notebook 4."
    )

cols_to_drop = sorted(
    set(leakage_columns + review_columns + id_columns + raw_timestamp_columns)
    & set(X_train.columns)
)
print("\nDropping leakage / review / ID / raw-timestamp columns:", cols_to_drop)

X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_validation = X_validation.drop(columns=cols_to_drop, errors="ignore")
X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

print("\nRemaining features for model input:")
for i, col in enumerate(X_train.columns, start=1):
    print(f"  {i:02d}. {col} ({X_train[col].dtype})")


✓ Loaded prediction-time feature contract from: /Users/ouahibaahmid/training mlops/artifacts/notebook_04/prediction_time_feature_table.csv

Dropping leakage / review / ID / raw-timestamp columns: ['average_review_score', 'customer_id', 'customer_unique_id', 'delivery_delay_days', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_id', 'order_purchase_timestamp', 'order_status', 'review_count']

Remaining features for model input:
  01. customer_zip_code_prefix (int64)
  02. customer_city (object)
  03. customer_state (object)
  04. item_count (float64)
  05. unique_products (float64)
  06. unique_sellers (float64)
  07. total_item_value (float64)
  08. total_freight_value (float64)
  09. payment_count (float64)
  10. total_payment_value (float64)
  11. max_payment_installments (float64)
  12. seller_zip_code_prefix (float64)
  13. seller_city (object)
  14. seller_state (object)
  15. distance_km (float64)
  16.

## 5. Build Preprocessing Pipelines

- **Zip Code Handling**: `customer_zip_code_prefix` / `seller_zip_code_prefix` are converted to categorical strings to prevent improper continuous scaling.
- **Numeric Features**: Imputed with the median (with a missingness indicator — Notebook 4 found some columns' missingness correlates with the target) and scaled with `StandardScaler`.
- **Categorical Features**: Imputed with constant `'missing'` and encoded with `OneHotEncoder`, using `min_frequency` / `max_categories` to group rare categories into an `"infrequent"` bucket. Notebook 4 explicitly warns against *blindly* one-hot-encoding high-cardinality columns like city or zip prefix, and recommends rare-category grouping over dropping — so genuinely informative high-cardinality columns are kept (with bounded width) instead of thrown away. Only columns that are effectively row-level identifiers (no repeated values to learn from) are dropped outright.


In [5]:
# Convert ZIP-code prefixes to string so they are treated as categorical
# identifiers rather than continuous numbers (a StandardScaler on a ZIP code
# would be meaningless).
for zip_col in ["customer_zip_code_prefix", "seller_zip_code_prefix"]:
    if zip_col in X_train.columns:
        X_train[zip_col] = X_train[zip_col].astype(str)
        X_validation[zip_col] = X_validation[zip_col].astype(str)
        X_test[zip_col] = X_test[zip_col].astype(str)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
all_categorical_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()

# High-cardinality categorical columns (city, zip prefix, ...) are NOT
# dropped outright. Notebook 4 explicitly flags these as prediction-time-safe
# and recommends "rare-category grouping ... rather than blindly one-hot
# encoding" high-cardinality columns — see the OneHotEncoder configuration
# below. The only columns dropped here are effective row-level identifiers
# (more unique values than half the rows), since grouping can't rescue a
# column with no repeated values to learn from.
IDENTIFIER_CARDINALITY_RATIO = 0.5
cardinality = X_train[all_categorical_features].nunique()
identifier_like_features = cardinality[
    cardinality > IDENTIFIER_CARDINALITY_RATIO * len(X_train)
].index.tolist()
categorical_features = [c for c in all_categorical_features if c not in identifier_like_features]

if identifier_like_features:
    print(f"Dropping identifier-like categorical columns (> {IDENTIFIER_CARDINALITY_RATIO:.0%} unique rows):")
    for col in identifier_like_features:
        print(f"  {col}: {cardinality[col]:,} unique values")
    X_train = X_train.drop(columns=identifier_like_features)
    X_validation = X_validation.drop(columns=identifier_like_features)
    X_test = X_test.drop(columns=identifier_like_features)

print("\nNumeric features (scaled):", numeric_features)
print("Categorical features (OHE + rare-category grouping):", categorical_features)
for col in categorical_features:
    if cardinality[col] > 30:
        print(f"  note: {col} has {cardinality[col]:,} unique values — infrequent "
              f"categories will be grouped into an 'infrequent' bucket below.")

# Numeric: median-impute, keeping a missingness indicator (add_indicator=True)
# since Notebook 4's missingness-vs-target analysis found informative
# missingness for some numeric columns — then standard-scale.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

# Categorical: constant-impute a 'missing' category, then one-hot-encode.
# min_frequency / max_categories implement rare-category grouping: infrequent
# categories (e.g. small cities) collapse into a single "infrequent" bucket
# instead of either exploding the encoded width or being dropped and losing
# all signal. handle_unknown="infrequent_if_exist" routes brand-new
# categories seen only at inference time into that same bucket instead of an
# all-zero row.
MAX_OHE_CATEGORIES = 30
MIN_CATEGORY_FREQUENCY = 0.01

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=MIN_CATEGORY_FREQUENCY,
        max_categories=MAX_OHE_CATEGORIES,
        sparse_output=True,
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)
print("\n✓ ColumnTransformer configured.")



Numeric features (scaled): ['item_count', 'unique_products', 'unique_sellers', 'total_item_value', 'total_freight_value', 'payment_count', 'total_payment_value', 'max_payment_installments', 'distance_km', 'estimated_delivery_days', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour', 'is_weekend']
Categorical features (OHE + rare-category grouping): ['customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
  note: customer_zip_code_prefix has 13,748 unique values — infrequent categories will be grouped into an 'infrequent' bucket below.
  note: customer_city has 3,747 unique values — infrequent categories will be grouped into an 'infrequent' bucket below.
  note: seller_zip_code_prefix has 1,681 unique values — infrequent categories will be grouped into an 'infrequent' bucket below.
  note: seller_city has 490 unique values — infrequent categories will be grouped into an 'infrequent' buck

## 6. Strict Zero-Leakage Fitting & Transformation

> **CRITICAL RULE**: 
> `preprocessor.fit()` is called **ONLY on `X_train`**.
> `X_validation` and `X_test` are transformed using the fitted transformer without calling `fit()`.

In [6]:
# FIT STRICTLY ON TRAIN
preprocessor.fit(X_train)
print("✓ Preprocessor fitted strictly on TRAIN dataset only.")

# Transform all splits
X_train_transformed = preprocessor.transform(X_train)
X_validation_transformed = preprocessor.transform(X_validation)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(f"Transformed shapes:")
print(f"  Train:      {X_train_transformed.shape}")
print(f"  Validation: {X_validation_transformed.shape}")
print(f"  Test:       {X_test_transformed.shape}")
print(f"  Total output features: {len(feature_names)}")

✓ Preprocessor fitted strictly on TRAIN dataset only.
Transformed shapes:
  Train:      (67533, 83)
  Validation: (14471, 83)
  Test:       (14472, 83)
  Total output features: 83


## 7. Persist Artifacts (joblib, npz, json)

We save the combined `preprocessor` (the single object production should
load), and — because "save every fitted object, not just the output table"
means each imputer/scaler/encoder individually — we also save each fitted
sub-object on its own so it can be inspected, audited, or reused
independently of the full pipeline.


In [7]:
# 1. Save fitted preprocessor (the single object production should load —
#    loading it never re-fits anything on new data)
preprocessor_path = artifact_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
print(f"✓ Saved combined preprocessor: {preprocessor_path.name}")

# 1b. Save each fitted sub-object individually as well, so every
#     encoder/scaler/imputer used is available on its own, not just bundled
#     inside the output table or the combined preprocessor.
numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
numeric_scaler = preprocessor.named_transformers_["numeric"].named_steps["scaler"]
categorical_imputer = preprocessor.named_transformers_["categorical"].named_steps["imputer"]
categorical_encoder = preprocessor.named_transformers_["categorical"].named_steps["onehot"]

fitted_objects = {
    "numeric_imputer.joblib": numeric_imputer,
    "numeric_scaler.joblib": numeric_scaler,
    "categorical_imputer.joblib": categorical_imputer,
    "categorical_encoder.joblib": categorical_encoder,
}
for filename, fitted_object in fitted_objects.items():
    joblib.dump(fitted_object, artifact_dir / filename)
print(f"✓ Saved individual fitted transformers: {', '.join(fitted_objects)}")

# 2. Save sparse feature matrices
X_train_sparse = sparse.csr_matrix(X_train_transformed) if not sparse.issparse(X_train_transformed) else X_train_transformed
X_val_sparse = sparse.csr_matrix(X_validation_transformed) if not sparse.issparse(X_validation_transformed) else X_validation_transformed
X_test_sparse = sparse.csr_matrix(X_test_transformed) if not sparse.issparse(X_test_transformed) else X_test_transformed

save_npz(artifact_dir / "X_train.npz", X_train_sparse)
save_npz(artifact_dir / "X_validation.npz", X_val_sparse)
save_npz(artifact_dir / "X_test.npz", X_test_sparse)
print("✓ Saved sparse feature matrices (.npz)")

# 3. Save target arrays
np.save(artifact_dir / "y_train.npy", y_train.to_numpy())
np.save(artifact_dir / "y_validation.npy", y_validation.to_numpy())
np.save(artifact_dir / "y_test.npy", y_test.to_numpy())
print("✓ Saved target arrays (.npy)")

# 4. Save feature config and names metadata
feature_config = {
    "target": TARGET,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "removed_leakage_columns": leakage_columns,
    "removed_review_columns": review_columns,
    "removed_id_columns": id_columns,
    "removed_raw_timestamp_columns": raw_timestamp_columns,
    "removed_identifier_like_columns": identifier_like_features,
    "feature_contract_source": (
        str(feature_contract_path) if feature_contract_path.exists() else "fallback_hardcoded"
    ),
    "one_hot_encoder_min_frequency": MIN_CATEGORY_FREQUENCY,
    "one_hot_encoder_max_categories": MAX_OHE_CATEGORIES,
    "numeric_imputer_add_indicator": True,
    "num_features_out": len(feature_names)
}

with open(artifact_dir / "feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

with open(artifact_dir / "feature_names.json", "w") as f:
    json.dump(feature_names.tolist(), f, indent=2)

pd.DataFrame({"feature_name": feature_names}).to_csv(artifact_dir / "feature_list.csv", index=False)
print("✓ Saved feature_config.json, feature_names.json, feature_list.csv")


✓ Saved combined preprocessor: preprocessor.joblib
✓ Saved individual fitted transformers: numeric_imputer.joblib, numeric_scaler.joblib, categorical_imputer.joblib, categorical_encoder.joblib
✓ Saved sparse feature matrices (.npz)
✓ Saved target arrays (.npy)
✓ Saved feature_config.json, feature_names.json, feature_list.csv


## 8. Artifact Reload Verification

A shape match alone doesn't prove a reloaded object is the same fitted
transformer, so we also compare actual transformed values. We additionally
confirm the individually-saved fitted objects, recomposed by hand, reproduce
the combined preprocessor's output — this is what "production loads the same
fitted objects, never fits again" needs to actually hold in practice if a
downstream service consumes the parts separately instead of the bundled
pipeline.


In [8]:
loaded_preprocessor = joblib.load(preprocessor_path)
val_check = loaded_preprocessor.transform(X_validation)
assert val_check.shape == X_validation_transformed.shape, "Shape mismatch after reload."

val_check_dense = val_check.toarray() if sparse.issparse(val_check) else val_check
val_original_dense = X_validation_transformed.toarray() if sparse.issparse(X_validation_transformed) else X_validation_transformed
np.testing.assert_allclose(val_check_dense, val_original_dense)
print("✓ Verification successful: reloaded preprocessor reproduces identical values (not just shape).")

# Confirm the individually-saved fitted objects, recomposed by hand, give the
# exact same result as the combined preprocessor.
reloaded_numeric_imputer = joblib.load(artifact_dir / "numeric_imputer.joblib")
reloaded_numeric_scaler = joblib.load(artifact_dir / "numeric_scaler.joblib")
reloaded_categorical_imputer = joblib.load(artifact_dir / "categorical_imputer.joblib")
reloaded_categorical_encoder = joblib.load(artifact_dir / "categorical_encoder.joblib")

manual_numeric = reloaded_numeric_scaler.transform(
    reloaded_numeric_imputer.transform(X_validation[numeric_features])
)
manual_categorical = reloaded_categorical_encoder.transform(
    reloaded_categorical_imputer.transform(X_validation[categorical_features])
)
manual_categorical_dense = manual_categorical.toarray() if sparse.issparse(manual_categorical) else manual_categorical
manual_combined = np.hstack([manual_numeric, manual_categorical_dense])
np.testing.assert_allclose(manual_combined, val_original_dense)
print("✓ Verification successful: individually-saved fitted objects reproduce the combined preprocessor's output.")


✓ Verification successful: reloaded preprocessor reproduces identical values (not just shape).
✓ Verification successful: individually-saved fitted objects reproduce the combined preprocessor's output.


## 9. Summary & Key Takeaways

1. **Leakage Prevention**: Leakage/review columns are pulled from Notebook 4's prediction-time feature contract (with a hard-coded fallback), so the two notebooks can't silently drift apart.
2. **Feature Engineering**: Engineered `estimated_delivery_days` and purchase calendar features prior to dropping raw dates.
3. **No Information Thrown Away Unnecessarily**: High-cardinality but prediction-time-safe columns (city, zip prefix) are kept via rare-category grouping (`OneHotEncoder(min_frequency=..., max_categories=...)`) instead of being dropped wholesale; only genuine row-level identifiers are dropped.
4. **Missingness Preserved**: Numeric imputation keeps a missingness indicator, since Notebook 4 found informative missingness for some columns.
5. **Zero-Leakage Guarantee**: Preprocessor fitted solely on `X_train`.
6. **Every Fitted Object Saved**: The combined `preprocessor.joblib` plus each fitted imputer/scaler/encoder individually — verified to reproduce identical output after reload, not just a matching shape.
7. **Artifacts Exported**: `preprocessor.joblib`, individual fitted transformers, `.npz` feature matrices, `.npy` targets, and the feature list — all in `artifacts/notebook_05/`, ready for model training in Notebook 6.
